# 13 - Intent Assurance

## Objetivo
Detalhar o pipeline de assurance de ponta a ponta - da evidência bruta
(`IntentEvidence`, eventos `DELIVERY`) até o veredito final
(`IntentEvaluation`, `Violation`) - com quatro casos reais que isolam cada
tipo de resultado: todas as condições passam, pares insuficientes,
fidelidade insuficiente, e nenhum par entregue.

## Conceitos
- `assurance.telemetry.collect_intent_evidence` constrói `IntentEvidence`
  exclusivamente a partir de eventos `EventTypes.DELIVERY` tagueados com
  `intent_id` - a única fonte de prova de entrega desta arquitetura.
- Deduplicação: cada `pair_number` só é contado uma vez por intent
  (`collect_intent_evidence` descarta duplicatas), e cada par fim a fim só
  é reportado no lado **iniciador** da reserva (`RequestApp.get_memory`
  incrementa `memory_counter` apenas quando `remote_node == responder`) -
  o mesmo par nunca é contado nos dois endpoints.
- `assurance.evaluator.evaluate_intent` produz um `ConditionResult` por
  condição declarada (`metric`, `operator`, `expected`, `observed`,
  `passed`, `evidence_source`), usando apenas o módulo `operator` -
  nunca `eval()`.
- `assurance.violations.classify_violations` converte `ConditionResult`s
  reprovados em `Violation`s tipados (categoria, gap relativo).
- A transição de lifecycle (`ACTIVE` -> `SATISFIED`/`VIOLATED`) só acontece
  **depois** dessa avaliação - nunca antes.

Por que métricas/estados brutos não bastam:
- **reserva aceita não implica intent satisfeito**: `ACTIVE` só significa
  que o `NetworkManager` aceitou negociar a reserva - diz nada sobre o que
  foi entregue depois.
- **métricas globais não servem como evidência por intent**: `eg_success`/
  `ep_success`/`es_success` são contadores por nó, não por intent (ver
  notebook 06 do H1).
- **`entangle_time > 0` não é suficiente**: não diferencia um par
  realmente casado e reportado de um estado transitório.
- **pares `PURIFIED` precisam ser contados**: o `RequestApp` padrão do
  SeQUeNCe ignora esse estado (ver notebook 03 do H1).
- **o mesmo par não pode ser contado nos dois endpoints**: só o lado
  iniciador da reserva conta um par como entregue.


## Configuração

Quatro intents sobre a mesma topologia `a - r - b`, isolando um tipo de resultado cada.


In [1]:
from ibqn.demos.topologies import three_node_spec
from ibqn.demos.intents import simple_intent
from ibqn.demos.scenarios import ad_hoc_scenario
from ibqn.experiments.runner import run_scenario
from ibqn.intent.models import SuccessCondition

SEED = 0
spec = three_node_spec(stop_time_s=0.15)  # shared by cases A/B/C

# Case A: everything a real, fast, low-attenuation run can deliver is enough.
intent_a = simple_intent(
    intent_id="case-a-satisfied", source="a", destination="b",
    min_fidelity=0.65, requested_pairs=10, duration=0.1,
)

# Case B: the network delivers plenty of pairs, but the declared success
# condition demands far more than any real window could produce.
intent_b = simple_intent(
    intent_id="case-b-insufficient-pairs", source="a", destination="b",
    min_fidelity=0.65, requested_pairs=10, duration=0.1,
    success_conditions=[SuccessCondition(metric="delivered_pairs", operator=">=", expected=100_000)],
)

# Case C: requirements.min_fidelity (planning) stays low so the plan is
# feasible; validation declares a much stricter average_fidelity bar than
# a no-purification swap can ever reach (see notebook 02 of H1: ~0.6864).
intent_c = simple_intent(
    intent_id="case-c-insufficient-fidelity", source="a", destination="b",
    min_fidelity=0.5, requested_pairs=10, duration=0.1,
    success_conditions=[SuccessCondition(metric="average_fidelity", operator=">=", expected=0.90)],
)

# Case D: the reservation window is far shorter than any single
# Barrett-Kok round trip can complete in - zero DELIVERY events, by
# construction of the timing, not by disabling anything.
intent_d = simple_intent(
    intent_id="case-d-no-pairs-delivered", source="a", destination="b",
    min_fidelity=0.65, requested_pairs=10, duration=1e-6,
    success_conditions=[
        SuccessCondition(metric="delivered_pairs", operator=">=", expected=1),
        SuccessCondition(metric="average_fidelity", operator=">=", expected=0.65),
    ],
)
spec_d = three_node_spec(stop_time_s=0.01)


## Execução

Cada caso roda numa reserva independente (`ad_hoc_scenario` + `run_scenario` reinicia `sequence.utils.metrics` a cada chamada).


In [2]:
results = {}
results["A"] = run_scenario(ad_hoc_scenario(spec, [intent_a], seed=SEED), seed=SEED).get(intent_a.id)
results["B"] = run_scenario(ad_hoc_scenario(spec, [intent_b], seed=SEED), seed=SEED).get(intent_b.id)
results["C"] = run_scenario(ad_hoc_scenario(spec, [intent_c], seed=SEED), seed=SEED).get(intent_c.id)
results["D"] = run_scenario(ad_hoc_scenario(spec_d, [intent_d], seed=SEED), seed=SEED).get(intent_d.id)

for label, r in results.items():
    print(f"Case {label}: final_status={r.final_status.value}, satisfied={r.satisfied}")


2026-07-11 15:49:49,962 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 3 routers, 2 quantum links, seed=0


2026-07-11 15:49:49,965 INFO     ibqn.ibqn.planning.planner intent_id=case-a-satisfied - planned route a->r->b, estimated_fidelity=0.6864, purification=False


2026-07-11 15:49:49,966 INFO     ibqn.ibqn.intent.repository intent_id=case-a-satisfied - intent registered


2026-07-11 15:49:49,966 INFO     ibqn.ibqn.intent.repository intent_id=case-a-satisfied - intent transitioned to VALIDATED (schema validated)


2026-07-11 15:49:49,967 INFO     ibqn.ibqn.intent.repository intent_id=case-a-satisfied - intent transitioned to PLANNING (invoking IntentPlanner)


...

2026-07-11 15:50:15,588 INFO     ibqn.ibqn.experiments.runner intent_id=- - running scenario 'ad-hoc-demo', seed=0, 1 intent(s)


2026-07-11 15:50:15,589 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.010s


2026-07-11 15:50:15,590 INFO     ibqn.ibqn.intent.repository intent_id=case-d-no-pairs-delivered - intent transitioned to ACTIVE (reservation approved)


2026-07-11 15:50:15,592 INFO     ibqn.ibqn.intent.repository intent_id=case-d-no-pairs-delivered - intent transitioned to VIOLATED (delivered_pairs >= 1.0 (observed=0.0) -> FAIL; average_fidelity >= 0.65 (observed=None) -> FAIL)


Case A: final_status=SATISFIED, satisfied=True
Case B: final_status=VIOLATED, satisfied=False
Case C: final_status=VIOLATED, satisfied=False
Case D: final_status=VIOLATED, satisfied=False


## Resultados


In [3]:
from ibqn.demos.tables import evaluation_table

for label in ["A", "B", "C", "D"]:
    print(f"\nCase {label} - {results[label].final_status.value}:")
    display(evaluation_table(results[label].evaluation))



Case A - SATISFIED:


,metric,operator,expected,observed,passed,evidence_source
0,delivered_pairs,>=,10.00,504.000000,True,504 DELIVERY event(s) tagged intent_id='case-a...
1,average_fidelity,>=,0.65,0.686375,True,504 DELIVERY event(s) tagged intent_id='case-a...



Case B - VIOLATED:


,metric,operator,expected,observed,passed,evidence_source
0,delivered_pairs,>=,100000.0,504.0,False,504 DELIVERY event(s) tagged intent_id='case-b...



Case C - VIOLATED:


,metric,operator,expected,observed,passed,evidence_source
0,average_fidelity,>=,0.9,0.686375,False,504 DELIVERY event(s) tagged intent_id='case-c...



Case D - VIOLATED:


,metric,operator,expected,observed,passed,evidence_source
0,delivered_pairs,>=,1.00,0.0,False,0 DELIVERY event(s) tagged intent_id='case-d-n...
1,average_fidelity,>=,0.65,NaN,False,0 DELIVERY event(s) tagged intent_id='case-d-n...


In [4]:
from ibqn.assurance.violations import classify_violations

for label in ["A", "B", "C", "D"]:
    violations = classify_violations(results[label].evaluation)
    print(f"Case {label}: {len(violations)} violation(s)")
    for v in violations:
        gap = f"{v.relative_gap:.2%}" if v.relative_gap is not None else "n/a"
        print(f"  category={v.category.value}, metric={v.metric}, expected={v.expected}, observed={v.observed}, relative_gap={gap}")


Case A: 0 violation(s)
Case B: 1 violation(s)
  category=delivered_pairs, metric=delivered_pairs, expected=100000.0, observed=504.0, relative_gap=99.50%
Case C: 1 violation(s)
  category=fidelity, metric=average_fidelity, expected=0.9, observed=0.6863749999999998, relative_gap=23.74%
Case D: 2 violation(s)
  category=delivered_pairs, metric=delivered_pairs, expected=1.0, observed=0.0, relative_gap=100.00%
  category=fidelity, metric=average_fidelity, expected=0.65, observed=None, relative_gap=n/a


In [5]:
assert results["A"].final_status.value == "SATISFIED"
assert all(r.passed for r in results["A"].evaluation.condition_results)

assert results["B"].final_status.value == "VIOLATED"
assert classify_violations(results["B"].evaluation)[0].category.value == "delivered_pairs"

assert results["C"].final_status.value == "VIOLATED"
assert classify_violations(results["C"].evaluation)[0].category.value == "fidelity"

assert results["D"].final_status.value == "VIOLATED"
d_conditions = {c.metric: c for c in results["D"].evaluation.condition_results}
assert d_conditions["delivered_pairs"].observed == 0
assert d_conditions["average_fidelity"].observed is None, "fidelity is undefined (not zero) when no pairs were ever delivered"
print("All four cases confirmed with the expected, isolated failure mode.")


All four cases confirmed with the expected, isolated failure mode.


## Interpretação
As quatro colunas de `evidence_source` mostram, para toda condição
avaliada, exatamente quantos eventos `DELIVERY` tagueados com aquele
`intent_id` sustentam o número observado - nunca um valor derivado de
`intent.requirements` ou de contadores globais por nó. O Caso D é
particularmente informativo: quando nenhum par é entregue,
`average_fidelity` não é reportado como `0.0` (o que sugeriria pares de
péssima qualidade) - é reportado como `None` (indefinido, não existe
evidência), e `evaluate_intent` trata isso corretamente como reprovação
sem tentar adivinhar um valor.

## Limitações
- Métricas suportadas por `assurance.evaluator` são um conjunto fixo
  (`delivered_pairs`, `average_fidelity`, `min_fidelity`, `throughput`,
  `completion_time`) - uma métrica fora dessa lista levanta
  `UnsupportedMetricError` em vez de ser silenciosamente ignorada (ver
  notebook 10, seção "métrica desconhecida").
- `throughput`/`completion_time` não foram exercitados diretamente aqui;
  ver `docs/assurance_design.md` para as fórmulas.

## Conclusão
Pipeline completo de assurance demonstrado com quatro casos reais e
isolados, confirmando que cada veredito (`SATISFIED`/`VIOLATED`) é
rastreável até evidência `DELIVERY` específica do intent, nunca até um
valor requisitado ou um contador global.
